Seuraava muistikirja on generoitu automaattisesti GitHub Copilot Chatin toimesta ja on tarkoitettu vain alkuasetuksiin


# Johdanto Prompt Engineeringiin
Prompt engineering on prosessi, jossa suunnitellaan ja optimoidaan kehotteita luonnollisen kielen käsittelytehtäviin. Se sisältää oikeiden kehotteiden valinnan, niiden parametrien hienosäädön ja suorituskyvyn arvioinnin. Prompt engineering on ratkaisevan tärkeää korkeiden tarkkuuksien ja tehokkuuden saavuttamiseksi NLP-malleissa. Tässä osiossa tutustumme prompt engineeringin perusteisiin käyttämällä OpenAI:n malleja tutkimukseen.


### Harjoitus 1: Tokenisointi
Tutustu tokenisointiin käyttäen tiktokenia, OpenAI:n avointa nopeaa tokenisointikirjastoa
Katso lisää esimerkkejä [OpenAI Cookbookista](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst).


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### Harjoitus 2: Vahvista OpenAI API -avaimen asennus

Suorita alla oleva koodi varmistaaksesi, että OpenAI-päätepisteesi on asetettu oikein. Koodi yrittää yksinkertaista peruskyselyä ja tarkistaa valmiin vastauksen. Syöte `oh say can you see` pitäisi täydentyä suunnilleen muotoon `by the dawn's early light..`


In [ ]:
# Uses the OpenAI client against the Azure OpenAI (Microsoft Foundry) v1 endpoint
# with the Responses API. See https://aka.ms/openai/start

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

def get_completion(prompt):
    response = client.responses.create(
        model=deployment,
        input=prompt,
        temperature=0, # this is the degree of randomness of the model's output
        max_output_tokens=1024,
        store=False,
    )
    return response.output_text

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)


### Harjoitus 3: Keinotekoiset lisäykset  
Tutki, mitä tapahtuu, kun pyydät LLM:ää tuottamaan vastauksia kehotteeseen, joka koskee aihetta, joka ei ehkä ole olemassa, tai aiheita, joista se ei välttämättä tiedä, koska ne ovat sen esikoulutusdatan ulkopuolella (uudemmat). Katso, miten vastaus muuttuu, jos kokeilet erilaista kehottetta tai eri mallia.  


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### Harjoitus 4: Ohjeisiin perustuva 
Käytä "text"-muuttujaa pääsisällön asettamiseen
ja "prompt"-muuttujaa ohjeen antamiseen, joka liittyy pääsisältöön.

Tässä pyydämme mallia tiivistämään tekstin toisen luokan oppilasta varten


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### Harjoitus 5: Monimutkainen kehotus 
Kokeile pyyntöä, jossa on järjestelmän, käyttäjän ja avustajan viestejä 
Järjestelmä asettaa avustajan kontekstin
Käyttäjän ja avustajan viestit tarjoavat monikierroksisen keskustelukontekstin

Huomaathan, että avustajan persoonallisuus on asetettu "sarkastiseksi" järjestelmän kontekstissa. 
Kokeile erilaista persoonallisuuskontekstia. Tai kokeile eri sarjaa syöte-/tulosviestejä


In [ ]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
    store=False,
)
print(response.output_text)


### Harjoitus: Tutki intuitiotasi
Yllä olevat esimerkit antavat sinulle malleja, joita voit käyttää uusien kehotteiden luomiseen (yksinkertaisia, monimutkaisia, ohjeistuksia jne.) – kokeile luoda muita harjoituksia tutkiaksesi joitakin muita ideoita, joista olemme puhuneet, kuten esimerkit, vihjeet ja muuta.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
